# SAM ONNX Export and CPU Runtime

This tutorial demonstrates how to export Segment Anything Model (SAM) to ONNX format
and run CPU inference using the VisionServeX runtime.

**Models covered:** `sam-vit-b`, `mobilesam`  
**Backend:** ONNX Runtime (CPU)  
**License:** Apache-2.0

In [ ]:
# Cell 2: Setup — install dependencies
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

pip_install('visionservex')
pip_install('onnxruntime')

import visionservex as vsx
import numpy as np
import time
print(f'VisionServeX version: {vsx.__version__}')

In [ ]:
# Cell 3: Export sam-vit-b to ONNX
import pathlib
from visionservex.onnx_export import list_onnx_eligible_models, export_sam_decoder_onnx

status = list_onnx_eligible_models()
out_dir = pathlib.Path('/tmp/vsx_v34_onnx')
out_dir.mkdir(parents=True, exist_ok=True)

sam_vit_b_path = out_dir / 'sam_vit_b_decoder.onnx'
s = next(x for x in status if x['model_id'] == 'sam-vit-b')

if s['checkpoint_exists']:
    result = export_sam_decoder_onnx('sam-vit-b', sam_vit_b_path)
    print(f'EXPORTED sam-vit-b: {result["size_mb"]} MB  opset={result["opset"]}')
else:
    print(f'SKIP sam-vit-b: checkpoint missing — run: visionservex pull sam-vit-b')
    print(f'  Expected path: {s["checkpoint_path"]}')

In [ ]:
# Cell 4: Run sam-vit-b ONNX on CPU, measure latency
from visionservex.onnx_export import run_sam_onnx_cpu

if sam_vit_b_path.exists():
    # Warm-up run
    _ = run_sam_onnx_cpu(str(sam_vit_b_path))

    # Timed runs (5 iterations)
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        rt = run_sam_onnx_cpu(str(sam_vit_b_path))
        times.append((time.perf_counter() - t0) * 1000)

    avg_ms = sum(times) / len(times)
    print(f'sam-vit-b  status={rt["status"]}  mask={rt["mask_shape"]}  iou={rt["iou_pred"]:.3f}')
    print(f'CPU latency (avg 5 runs): {avg_ms:.1f} ms')
    print(f'Decoder latency (internal): {rt["decoder_latency_ms"]} ms')
else:
    print('sam-vit-b ONNX not found — export step required first')

In [ ]:
# Cell 5: Show result summary
if sam_vit_b_path.exists():
    print('sam-vit-b ONNX CPU result')
    print(f'  Status       : {rt["status"]}')
    print(f'  Mask shape   : {rt["mask_shape"]}')
    print(f'  IoU pred     : {rt["iou_pred"]:.3f}')
    print(f'  Avg latency  : {avg_ms:.1f} ms')
    print(f'  ONNX size    : {sam_vit_b_path.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Cell 6: Export MobileSAM to ONNX
mobile_path = out_dir / 'mobilesam_decoder.onnx'
ms = next(x for x in status if x['model_id'] == 'mobilesam')

if ms['checkpoint_exists']:
    mobile_result = export_sam_decoder_onnx('mobilesam', mobile_path)
    print(f'EXPORTED mobilesam: {mobile_result["size_mb"]} MB  opset={mobile_result["opset"]}')
else:
    print(f'SKIP mobilesam: checkpoint missing — run: visionservex pull mobilesam')
    print(f'  Expected path: {ms["checkpoint_path"]}')

In [ ]:
# Cell 7: Run MobileSAM ONNX on CPU
if mobile_path.exists():
    _ = run_sam_onnx_cpu(str(mobile_path))  # warm-up
    mobile_times = []
    for _ in range(5):
        t0 = time.perf_counter()
        mrt = run_sam_onnx_cpu(str(mobile_path))
        mobile_times.append((time.perf_counter() - t0) * 1000)
    mobile_avg = sum(mobile_times) / len(mobile_times)
    print(f'mobilesam  status={mrt["status"]}  mask={mrt["mask_shape"]}  iou={mrt["iou_pred"]:.3f}')
    print(f'CPU latency (avg 5 runs): {mobile_avg:.1f} ms')
else:
    mobile_avg = None
    print('mobilesam ONNX not found — export step required first')

In [ ]:
# Cell 8: Comparison table
rows = []
if sam_vit_b_path.exists():
    rows.append(('sam-vit-b',  'ONNX CPU', f'{avg_ms:.1f}',       rt["iou_pred"],    'ready'))
if mobile_path.exists():
    rows.append(('mobilesam',  'ONNX CPU', f'{mobile_avg:.1f}',   mrt["iou_pred"],   'ready'))

print(f'{"Model":<15} {"Backend":<12} {"Latency ms":<14} {"IoU":<8} {"Status"}')
print('-' * 60)
for r in rows:
    print(f'{r[0]:<15} {r[1]:<12} {r[2]:<14} {r[3]:<8.3f} {r[4]}')

if len(rows) == 2:
    speedup = avg_ms / mobile_avg if mobile_avg else float('nan')
    print(f'\nMobileSAM speedup vs sam-vit-b: {speedup:.2f}x')

## Next Steps

- See `02_sam21_promptable_benchmark.ipynb` for SAM 2.1 promptable segmentation.
- To run on GPU, install `onnxruntime-gpu` and pass `device='cuda'`.
- For video tracking see `03_sam21_video_tracking.ipynb`.